In [66]:
# filter dataset
!python3 MT-Preparation/filtering/filter.py ./en-zh.en ./en-zh.zh en zh

Dataframe shape (rows, columns): (231267, 2)
--- Rows with Empty Cells Deleted	--> Rows: 231267
--- Duplicates Deleted			--> Rows: 229646
--- Source-Copied Rows Deleted		--> Rows: 229640
--- Too Long Source/Target Deleted	--> Rows: 224743
--- HTML Removed			--> Rows: 224743
--- Rows will remain true-cased		--> Rows: 224743
--- Rows with Empty Cells Deleted	--> Rows: 224743
--- Source Saved: ./en-zh.en-filtered.en
--- Target Saved: ./en-zh.zh-filtered.zh


In [91]:
import spacy

# Load English model
nlp = spacy.load("en_core_web_sm")

def tag_file(input_file, output_file):
    with open(input_file, "r", encoding="utf-8") as infile, \
         open(output_file, "w", encoding="utf-8") as outfile:
        # Read all lines, preserving empty ones
        lines = infile.readlines()
        # Process each line individually
        for line in lines:
            line = line.strip()
            if not line:  # Handle empty lines
                outfile.write("\n")
                continue
            doc = nlp(line)
            tagged_line = " ".join(f"{token.text}_{token.pos_}" for token in doc)
            outfile.write(tagged_line + "\n")

# Tag your English file
tag_file("en-zh.en-filtered.en", "en-zh_tagged.en")

In [83]:
!pip install sentencepiece

In [84]:
# train a sentencepiece model for subwording
!python3 MT-Preparation/subwording/1-train_unigram.py ./en-zh_tagged.en ./en-zh.zh-filtered.zh 

sentencepiece_trainer.cc(178) LOG(INFO) Running command: --input=./en-zh_tagged.en --model_prefix=source --vocab_size=20000 --hard_vocab_limit=false --split_digits=true --user_defined_symbols=__SEP__--byte_fallback=true
sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: ./en-zh_tagged.en
  input_format: 
  model_prefix: source
  model_type: UNIGRAM
  vocab_size: 20000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 1
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  user_defined_symbols: __SEP__--byte_fallback=true
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_e

In [85]:
# subword the dataset
!python3 MT-Preparation/subwording/2-subword.py source.model target.model ./en-zh_tagged.en ./en-zh.zh-filtered.zh

Source Model: source.model
Target Model: target.model
Source Dataset: ./en-zh_tagged.en
Target Dataset: ./en-zh.zh-filtered.zh
Done subwording the source file! Output: ./en-zh_tagged.en.subword
Done subwording the target file! Output: ./en-zh.zh-filtered.zh.subword


In [86]:
# first 3 lines before subwording
!head -n 3 ./en-zh_tagged.en && echo "-----" && head -n 3 ./en-zh.zh-filtered.zh

en_VERB
Thank_VERB you_PRON so_ADV much_ADV ,_PUNCT Chris_PROPN ._PUNCT And_CCONJ it_PRON 's_AUX truly_ADV a_DET great_ADJ honor_NOUN to_PART have_VERB the_DET opportunity_NOUN to_PART come_VERB to_ADP this_DET stage_NOUN twice_ADV ;_PUNCT I_PRON 'm_AUX extremely_ADV grateful_ADJ ._PUNCT
I_PRON have_AUX been_AUX blown_VERB away_ADV by_ADP this_DET conference_NOUN ,_PUNCT and_CCONJ I_PRON want_VERB to_PART thank_VERB all_PRON of_ADP you_PRON for_ADP the_DET many_ADJ nice_ADJ comments_NOUN about_ADP what_PRON I_PRON had_AUX to_PART say_VERB the_DET other_ADJ night_NOUN ._PUNCT
-----
zh
非常谢谢，克里斯。的确非常荣幸 能有第二次站在这个台上的机会，我真是非常感激。
这个会议真是让我感到惊叹不已，我还要谢谢你们留下的 关于我上次演讲的精彩评论


In [87]:
# first 3 lines after subwording
!head -n 3 ./en-zh_tagged.en.subword && echo "---" && head -n 3 ./en-zh.zh-filtered.zh.subword 

▁ en _ VERB
▁Thank _ VERB ▁you _ PRON ▁so _ ADV ▁muc h _ ADV ▁,_ PUNCT ▁Chris _ PROPN ▁._ PUNCT ▁And _ CCONJ ▁it _ PRON ▁' s _ AUX ▁tru ly _ ADV ▁a _ DET ▁great _ ADJ ▁honor _ NOUN ▁to _ P ART ▁have _ VERB ▁the _ DET ▁opportunity _ NOUN ▁to _ P ART ▁come _ VERB ▁to _ ADP ▁thi s _ DET ▁stage _ NOUN ▁twi ce _ ADV ▁ ; _ PUNCT ▁I _ PRON ▁' m _ AUX ▁extreme ly _ ADV ▁grateful _ ADJ ▁._ PUNCT
▁I _ PRON ▁have _ AUX ▁be en _ AUX ▁blow n _ VERB ▁away _ ADV ▁by _ ADP ▁thi s _ DET ▁conference _ NOUN ▁,_ PUNCT ▁and _ CCONJ ▁I _ PRON ▁want _ VERB ▁to _ P ART ▁thank _ VERB ▁all _ PRON ▁of _ ADP ▁you _ PRON ▁for _ ADP ▁the _ DET ▁many _ ADJ ▁nice _ ADJ ▁comment s _ NOUN ▁about _ ADP ▁what _ PRON ▁I _ PRON ▁had _ AUX ▁to _ P ART ▁say _ VERB ▁the _ DET ▁other _ ADJ ▁night _ NOUN ▁._ PUNCT
---
▁ z h
▁非常 谢谢 , 克里斯 。 的确 非常 荣幸 ▁能 有 第二次 站在 这个 台上 的机会 , 我 真是 非常 感激 。
▁这个 会议 真是 让我 感到 惊 叹 不 已 , 我 还要 谢谢你们 留下 的 ▁关于 我 上 次 演讲 的 精彩 评论


In [93]:
# split the dataset into training set, development set, and test set
# Development and test sets should be between 1000 and 5000 segments (here we chose 200)
!python3 MT-Preparation/train_dev_split/train_dev_test_split.py 2000 2000 ./en-zh_tagged.en.subword ./en-zh.zh-filtered.zh.subword

Dataframe shape: (224743, 2)
--- Empty Cells Deleted --> Rows: 224743
--- Wrote Files
Done!
Output files
./en-zh_tagged.en.subword.train
./en-zh.zh-filtered.zh.subword.train
./en-zh_tagged.en.subword.dev
./en-zh.zh-filtered.zh.subword.dev
./en-zh_tagged.en.subword.test
./en-zh.zh-filtered.zh.subword.test


In [94]:
!wc -l ./en-zh.*
!wc -l ./en-zh_tagged.*

  231267 ./en-zh.en
  224743 ./en-zh.en-filtered.en
  231267 ./en-zh.zh
  224743 ./en-zh.zh-filtered.zh
  224743 ./en-zh.zh-filtered.zh.subword
    2000 ./en-zh.zh-filtered.zh.subword.dev
    2000 ./en-zh.zh-filtered.zh.subword.test
  220743 ./en-zh.zh-filtered.zh.subword.train
 1361506 total
  224743 ./en-zh_tagged.en
  224743 ./en-zh_tagged.en.subword
    2000 ./en-zh_tagged.en.subword.dev
    2000 ./en-zh_tagged.en.subword.test
  220743 ./en-zh_tagged.en.subword.train
  674229 total


In [95]:
!wc -l ./*.subword.*

    2000 ./en-zh.zh-filtered.zh.subword.dev
    2000 ./en-zh.zh-filtered.zh.subword.test
  220743 ./en-zh.zh-filtered.zh.subword.train
    2000 ./en-zh_tagged.en.subword.dev
    2000 ./en-zh_tagged.en.subword.test
  220743 ./en-zh_tagged.en.subword.train
  449486 total


In [96]:
# check the first and last line from each dataset
!echo "---First line---"
!head -n 1 ./*.{train,dev,test}

!echo -e "\n---Last line---"
!tail -n 1 ./*.{train,dev,test}

---First line---
==> ./en-zh.zh-filtered.zh.subword.train <==
▁ z h

==> ./en-zh_tagged.en.subword.train <==
▁ en _ VERB

==> ./en-zh.zh-filtered.zh.subword.dev <==
▁所以 这两 样东西 是 联合 起来 的 。 ▁其实 就是 你的 受 教育 程度 和 周围 邻居 的 类型 , ▁我们 一会儿 再 具体 的 谈 一 谈 。

==> ./en-zh_tagged.en.subword.dev <==
▁So _ ADV ▁it _ PRON ▁' s _ AUX ▁the _ DET ▁combination _ NOUN ▁of _ ADP ▁these _ DET ▁two _ NUM ▁thing s _ NOUN ▁:_ PUNCT ▁it _ PRON ▁' s _ AUX ▁education _ NOUN ▁and _ CCONJ ▁the _ DET ▁type _ NOUN ▁of _ ADP ▁neighbor s _ NOUN ▁that _ PRON ▁you _ PRON ▁have _ VERB ▁,_ PUNCT ▁which _ PRON ▁we _ PRON ▁' ll _ AUX ▁talk _ VERB ▁about _ ADP ▁more _ ADJ ▁in _ ADP ▁a _ DET ▁moment _ NOUN ▁._ PUNCT

==> ./en-zh.zh-filtered.zh.subword.test <==
▁在 捷 克斯 洛 伐 克 , 东 德 ▁ 爱 沙 尼亚 , 拉 脱 维 亚 , 立 陶 宛 , ▁ 马 里 , 马 达 加 斯 加 , ▁ 波 兰 , 菲 律 宾 , ▁ 塞 尔 维 亚 , 斯 洛 维 尼亚 的 独裁 政府 , 我可以 继续 , ▁还有 现在 的 突 尼 斯 和 埃及 。

==> ./en-zh_tagged.en.subword.test <==
▁Dictatorship s _ NOUN ▁in _ ADP ▁Czech os lov aki a _ PROPN ▁,_ PUNCT ▁East _ PROPN ▁Ger